# Assignment 7 Example

This notebook gives one complete, minimal solution to the FX assignment.

It covers:
- Part A: a basic-balance strategy for the trade-weighted dollar index
- Part B: a simple carry-trade strategy across the six currencies in the workbook

The logic is intentionally simple so you can follow it and then reimplement it yourself in `assignment_7.ipynb`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.float_format', '{:,.4f}'.format)

if (Path.cwd() / 'data').exists() and (Path.cwd() / 'notebooks').exists():
    PROJECT_DIR = Path.cwd()
else:
    PROJECT_DIR = Path.cwd().parent

DATA_DIR = PROJECT_DIR / 'data'
BASIC_BALANCE_PATH = DATA_DIR / 'assignment_basic_balance_data.xlsx'
CARRY_PATH = DATA_DIR / 'assignment_carry_data.xlsx'

BASIC_BALANCE_PATH, CARRY_PATH

## Part A: Basic Balance

The assignment gives the Deutsche Bank relationship

`ln(TWI_t) = 4.54 + 0.000871 * BasicBalance_(t-T)`

For each lag `T`, the signal is:
- long the dollar if predicted log TWI is above the actual log TWI
- short the dollar otherwise

I measure the trade using the next one-month TWI return.

In [ ]:
bb = pd.read_excel(BASIC_BALANCE_PATH, sheet_name='Basic Balance').rename(
    columns={
        'Unnamed: 0': 'Date',
        'Ln(TWI)': 'LnTWI',
        'Basic Balance ': 'BasicBalance',
    }
)

bb['Date'] = pd.to_datetime(bb['Date'])
for col in ['TWI', 'LnTWI', 'BasicBalance']:
    bb[col] = pd.to_numeric(bb[col], errors='coerce')

bb = bb[['Date', 'TWI', 'LnTWI', 'BasicBalance']].copy()
bb['TWI_ret_1m'] = bb['TWI'].pct_change().shift(-1)

bb.head()

In [ ]:
def annualized_sharpe(returns, periods_per_year):
    clean = returns.dropna()
    if clean.empty or clean.std(ddof=1) == 0:
        return np.nan
    return np.sqrt(periods_per_year) * clean.mean() / clean.std(ddof=1)


def run_basic_balance_strategy(data, lag_months, threshold=0.0):
    frame = data.copy()
    frame['predicted_LnTWI'] = 4.54 + 0.000871 * frame['BasicBalance'].shift(lag_months)
    frame['gap'] = frame['predicted_LnTWI'] - frame['LnTWI']

    signal = np.where(frame['gap'] > threshold, 1, np.where(frame['gap'] < -threshold, -1, 0))
    frame['signal'] = pd.Series(signal, index=frame.index, dtype='float64')

    valid = frame[['predicted_LnTWI', 'gap', 'TWI_ret_1m']].notna().all(axis=1)
    frame.loc[~valid, 'signal'] = np.nan
    frame['strategy_ret'] = frame['signal'] * frame['TWI_ret_1m']
    frame['equity'] = (1 + frame['strategy_ret'].fillna(0)).cumprod()

    clean = frame['strategy_ret'].dropna()
    summary = {
        'obs': int(clean.shape[0]),
        'avg_monthly_ret': clean.mean(),
        'vol_monthly': clean.std(ddof=1),
        'sharpe_annual': annualized_sharpe(clean, 12),
        'cum_return': (1 + clean).prod() - 1,
        'active_share': frame['signal'].dropna().ne(0).mean(),
    }
    return frame, summary


bb_results = {lag: run_basic_balance_strategy(bb, lag_months=lag, threshold=0.0) for lag in range(1, 7)}

bb_summary = pd.DataFrame(
    [
        {'lag_months': lag, **result[1]}
        for lag, result in bb_results.items()
    ]
).sort_values('lag_months').reset_index(drop=True)

bb_summary

In [ ]:
best_raw_lag = int(bb_summary.sort_values('sharpe_annual', ascending=False).iloc[0]['lag_months'])
best_tradable_lag = int(
    bb_summary.loc[bb_summary['lag_months'] >= 2]
    .sort_values('sharpe_annual', ascending=False)
    .iloc[0]['lag_months']
)

print(f'Best raw lag in this sample: T={best_raw_lag}')
print(f'Best tradable lag after excluding T=1: T={best_tradable_lag}')

fig, ax = plt.subplots(figsize=(10, 5))
for lag in range(1, 7):
    frame, _ = bb_results[lag]
    ax.plot(frame['Date'], frame['equity'], label=f'T={lag}')
ax.set_title('Basic-Balance Strategy Equity Curves by Lag')
ax.set_ylabel('Growth of $1')
ax.legend(ncol=3)
plt.show()

In [ ]:
thresholds = np.round(np.arange(0.00, 0.051, 0.01), 2)

bb_threshold_results = {
    threshold: run_basic_balance_strategy(bb, lag_months=best_tradable_lag, threshold=threshold)
    for threshold in thresholds
}

bb_threshold_summary = pd.DataFrame(
    [
        {'threshold': threshold, **result[1]}
        for threshold, result in bb_threshold_results.items()
    ]
).sort_values('threshold').reset_index(drop=True)

best_threshold = float(bb_threshold_summary.sort_values('sharpe_annual', ascending=False).iloc[0]['threshold'])
print(f'Best threshold for the tradable lag in this sample: {best_threshold:.2f}')

bb_threshold_summary

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for threshold in [0.00, 0.02, 0.05]:
    frame, _ = bb_threshold_results[round(threshold, 2)]
    ax.plot(frame['Date'], frame['equity'], label=f'threshold={threshold:.2f}')
ax.set_title(f'Basic-Balance Equity Curves for T={best_tradable_lag}')
ax.set_ylabel('Growth of $1')
ax.legend()
plt.show()

## Part B: Carry Trade

For each currency, compare the foreign policy rate with the US policy rate.

Signal by currency:
- long the foreign currency if `foreign rate - US rate > threshold`
- short the foreign currency if `foreign rate - US rate < -threshold`
- otherwise stay flat

I then equal-weight the six currency signals into one carry basket.

In [ ]:
def parse_mixed_excel_date(value):
    if pd.isna(value):
        return pd.NaT
    if isinstance(value, pd.Timestamp):
        return value
    text = str(value).split('.')[0]
    if text.isdigit() and len(text) == 8:
        return pd.to_datetime(text, format='%Y%m%d')
    return pd.to_datetime(value, errors='coerce')


carry_raw = pd.read_excel(CARRY_PATH, sheet_name='Sheet1', header=1)
carry = carry_raw.iloc[1:].copy().reset_index(drop=True)

carry = carry.rename(
    columns={
        'Unnamed: 0': 'Date',
        'Unnamed: 1': 'US',
        'Unnamed: 2': 'Brit',
        'Unnamed: 3': 'CanadaRate',
        'Unnamed: 4': 'EuroRate',
        'Unnamed: 5': 'JapanRate',
        'Unnamed: 6': 'SwissRate',
        'Unnamed: 7': 'AussieRate',
    }
)

carry['Date'] = carry['Date'].apply(parse_mixed_excel_date)

numeric_cols = [
    'US', 'Brit', 'CanadaRate', 'EuroRate', 'JapanRate', 'SwissRate', 'AussieRate',
    'British', 'Canada', 'URO', 'Japan', 'Swiss', 'Aussie',
]
for col in numeric_cols:
    carry[col] = pd.to_numeric(carry[col], errors='coerce')

carry = carry[['Date'] + numeric_cols].dropna(subset=['Date']).reset_index(drop=True)
carry.head()

In [ ]:
currency_map = {
    'BP': ('Brit', 'British'),
    'CANADA': ('CanadaRate', 'Canada'),
    'EURO': ('EuroRate', 'URO'),
    'JAPAN': ('JapanRate', 'Japan'),
    'SWISS': ('SwissRate', 'Swiss'),
    'AUSSIE': ('AussieRate', 'Aussie'),
}


def run_carry_strategy(data, threshold=0.0):
    frame = data.copy()
    return_cols = []
    signal_cols = []

    for name, (rate_col, ret_col) in currency_map.items():
        diff_col = f'diff_{name}'
        signal_col = f'signal_{name}'
        ret_strategy_col = f'ret_{name}'

        frame[diff_col] = frame[rate_col] - frame['US']
        signal = np.where(frame[diff_col] > threshold, 1, np.where(frame[diff_col] < -threshold, -1, 0))
        frame[signal_col] = pd.Series(signal, index=frame.index, dtype='float64')

        valid = frame[[rate_col, ret_col, 'US']].notna().all(axis=1)
        frame.loc[~valid, signal_col] = np.nan
        frame[ret_strategy_col] = frame[signal_col] * (frame[ret_col] / 100.0)

        return_cols.append(ret_strategy_col)
        signal_cols.append(signal_col)

    frame['basket_ret'] = frame[return_cols].mean(axis=1, skipna=True)
    frame['equity'] = (1 + frame['basket_ret'].fillna(0)).cumprod()
    frame['active_share'] = frame[signal_cols].abs().mean(axis=1, skipna=True)

    clean = frame['basket_ret'].dropna()
    summary = {
        'avg_active_share': frame['active_share'].dropna().mean(),
        'avg_daily_ret': clean.mean(),
        'daily_vol': clean.std(ddof=1),
        'sharpe_annual': annualized_sharpe(clean, 252),
        'cum_return': (1 + clean).prod() - 1,
    }
    return frame, summary


carry_thresholds = np.round(np.arange(0.0, 0.51, 0.1), 1)
carry_results = {threshold: run_carry_strategy(carry, threshold=threshold) for threshold in carry_thresholds}

carry_summary = pd.DataFrame(
    [
        {'threshold_pct_pts': threshold, **result[1]}
        for threshold, result in carry_results.items()
    ]
).sort_values('threshold_pct_pts').reset_index(drop=True)

best_carry_threshold = float(carry_summary.sort_values('sharpe_annual', ascending=False).iloc[0]['threshold_pct_pts'])
print(f'Best carry threshold in this sample: {best_carry_threshold:.1f} percentage points')

carry_summary

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for threshold in [0.0, 0.3, 0.5]:
    frame, _ = carry_results[round(threshold, 1)]
    ax.plot(frame['Date'], frame['equity'], label=f'threshold={threshold:.1f}')
ax.set_title('Carry Basket Equity Curves by Threshold')
ax.set_ylabel('Growth of $1')
ax.legend()
plt.show()

## Final Takeaways

- In this sample, the strongest raw basic-balance lag is `T=1`, but that is not tradable because of the release delay in the assignment prompt.
- After excluding `T=1`, the best tradable lag is `T=2`, not `T=6`. So this sample does not support the claim that the six-month lag is the best predictor.
- A small threshold can help the basic-balance rule a little by filtering weak signals, but a very large threshold leaves you out of the market too often.
- The carry basket is positive in this sample, but thresholding does not improve it much.
- For a short assignment submission, the key is to show the rule, the comparison table, and a brief conclusion about which lag or threshold worked best.